#Lección 1: Fundamentos de Big Data

1. Las 5V de Big Data en RetailMax

**Volumen: **Procesamiento de millones de transacciones diarias, registros de navegación y miles de reseñas de productos.

**Velocidad:** Necesidad de analizar el comportamiento de navegación y transacciones en tiempo real para ofrecer recomendaciones personalizadas.

**Variedad:** Manejo de datos estructurados (transacciones SQL), semi-estructurados (logs de navegación) y no estructurados (reseñas de texto).

**Veracidad:** Limpieza y validación de datos provenientes de sistemas fragmentados para asegurar la calidad de los modelos de ML.

**Valor: **Conversión de datos masivos en información accionable para optimizar campañas de marketing y clasificar usuarios.

2. Fuentes de Datos y Arquitectura InicialFuentes: Transacciones de compra, logs de navegación del e-commerce y calificaciones de usuarios.Arquitectura Propuesta: Ingesta de datos mediante procesos distribuidos, procesamiento central en Apache Spark (RDDs y DataFrames), almacenamiento optimizado en formato Parquet y modelado predictivo con Spark MLlib.

#Lección 2: Apache Spark - Introducción y Configuración

In [ ]:
# Importación de librerías necesarias
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Configurar SparkContext y SparkSession
conf = SparkConf().setAppName("RetailMax_Leccion2").setMaster("local[*]")
sc = SparkContext(conf=conf)
spark = SparkSession.builder.config(conf=conf).getOrCreate()

# Cargar datos iniciales en RDDs y explorar acciones básicas
# Simulando carga de un dataset masivo de transacciones
data = [("TX100", 250.5), ("TX101", 10.0), ("TX102", 450.0), ("TX103", 5.0)]
rdd_inicial = sc.parallelize(data)

# Validar conectividad con acciones básicas
print(f"Total de registros cargados: {rdd_inicial.count()}")
print(f"Muestra de datos: {rdd_inicial.take(2)}")

#Lección 3: Elementos básicos de Spark (RDD, Transformaciones y Acciones)

In [ ]:
#Crear RDDs y Pair RDDs a partir de datos de transacciones [cite: 52]
# Estructura: (ID_Usuario, (Monto, Categoria))
transacciones_raw = [
    ("U1", (100.0, "Electro")), ("U2", (50.0, "Ropa")),
    ("U1", (150.0, "Electro")), ("U3", (200.0, "Hogar")),
    ("U2", (30.0, "Ropa")), ("U1", (20.0, "Electro"))
]
rdd_transacciones = sc.parallelize(transacciones_raw)

# Aplicar transformaciones (map, filter, distinct)
# 1. Filtrar solo categoría "Electro"
electro_rdd = rdd_transacciones.filter(lambda x: x[1][1] == "Electro")

# 2. Mapear para obtener solo montos por usuario
montos_usuario = electro_rdd.map(lambda x: (x[0], x[1][0]))

# 3. Obtener usuarios distintos que compraron Electro
usuarios_distinct = electro_rdd.map(lambda x: x[0]).distinct()

#Ejecutar acciones y documentar linaje
total_electro = montos_usuario.map(lambda x: x[1]).sum() # Acción: sum
print(f"Suma total en Electro: {total_electro}")

# El DAG (Grafo Acíclico Dirigido) se genera automáticamente al ejecutar acciones
print("Linaje del RDD final:")
print(montos_usuario.toDebugString().decode())

#Lección 4: Procesamiento de datos estructurados (Spark SQL y DataFrames)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

#Transformar RDDs a DataFrames con esquemas explícitos
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("monto", DoubleType(), True),
    StructField("categoria", StringType(), True)
])

# Preparar datos para el DataFrame
df_data = rdd_transacciones.map(lambda x: (x[0], x[1][0], x[1][1]))
df_retail = spark.createDataFrame(df_data, schema)

# Ejecutar consultas SQL para métricas de negocio
df_retail.createOrReplaceTempView("ventas_retail")

# Métricas: Ventas totales por categoría
metricas_categoria = spark.sql("""
    SELECT categoria, SUM(monto) as total_ventas, COUNT(*) as cantidad_tx
    FROM ventas_retail
    GROUP BY categoria
    ORDER BY total_ventas DESC
""")
metricas_categoria.show()

#Guardar resultados en formato Parquet
metricas_categoria.write.mode("overwrite").parquet("outputs/metricas_retail.parquet")

#Lección 5: Introducción a Machine Learning Escalable (Spark MLlib)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Cargar DataFrames procesados y preparar features
df_ml = spark.read.parquet("outputs/metricas_retail.parquet")

# Indexar categorías (String a Numérico)
indexer = StringIndexer(inputCol="categoria", outputCol="categoria_idx")
df_indexed = indexer.fit(df_ml).transform(df_ml)

# Crear vector de características (features)
assembler = VectorAssembler(inputCols=["total_ventas", "cantidad_tx"], outputCol="features")
final_data = assembler.transform(df_indexed)

#Entrenar modelos Supervisado (LR) y No Supervisado (K-Means)
# 1. K-Means para segmentación de usuarios
kmeans = KMeans(k=3, seed=1, featuresCol="features", predictionCol="segmento")
model_km = kmeans.fit(final_data)
segmentos = model_km.transform(final_data)

# 2. Regresión Logística (Simulando clasificación de propensión)
lr = LogisticRegression(featuresCol="features", labelCol="categoria_idx")
model_lr = lr.fit(final_data)

# Evaluar métricas y reporte
# El pipeline permite al área de marketing identificar clusters de comportamiento masivo.
print("Pipeline de ML finalizado. Resultados listos para reporte final en PDF.")
segmentos.select("categoria", "total_ventas", "segmento").show()